In [ ]:
from pathlib import Path

from vectormesh.data.cache import VectorCache

artefacts = Path("../artefacts")
trainpath = next(artefacts.glob("*bert*train/"))
validpath = next(artefacts.glob("*bert*valid/"))
traincache = VectorCache.load(path=trainpath)
validcache = VectorCache.load(path=validpath)


We load all data

In [ ]:
from vectormesh.data import OneHot

onehot = OneHot(num_classes=32, label_col="labels", target_col="onehot")
train_oh = traincache.dataset.map(onehot)
valid_oh = validcache.dataset.map(onehot)

Set the output label to a onehot encoded vector, and add fixed padding to the dataloader.

In [ ]:
from torch.utils.data import DataLoader

from vectormesh.components import FixedPadding
from vectormesh.data import Collate

collate_fn = Collate(
    embedding_col="legal_dutch",
    target_col="onehot",
    padder=FixedPadding(max_chunks=30),
)

trainloader = DataLoader(train_oh, batch_size=32, shuffle=True, collate_fn=collate_fn)
validloader = DataLoader(valid_oh, batch_size=32, shuffle=False, collate_fn=collate_fn)

A Mixture-of-Experts layer replaces a single dense feed-forward block with many independent "expert" sub-networks plus a small trainable gating/router network. See the paper `Outrageously Large Neural Networks` in the `references/` folder for more details.

In [ ]:
from vectormesh.components import MaskedMeanAggregator, NeuralNet, Serial
from vectormesh.components.gating import MoE

moe = MoE(
    experts=[
        NeuralNet(hidden_size=768, out_size=32),
        NeuralNet(hidden_size=768, out_size=32),
        NeuralNet(hidden_size=768, out_size=32),
        NeuralNet(hidden_size=768, out_size=32),
    ],
    hidden_size=768,
    out_size=32,
)

pipeline = Serial([MaskedMeanAggregator(), moe])

Obviously, this can be improved (see notebook `2_design` for inspiration) but that is left as an exercise for the reader.

In [ ]:
import torch
import torch.optim as optim
from mltrainer import ReportTypes, Trainer, TrainerSettings

from vectormesh.components.metrics import F1Score
from vectormesh.data.vectorizers import detect_device

device = detect_device()
print(f"Using device: {device}")

log_dir = Path("tmp").absolute()

settings = TrainerSettings(
    epochs=3,
    metrics=[F1Score()],
    logdir=log_dir,
    train_steps=len(trainloader),
    valid_steps=len(trainloader),
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)

loss_fn = torch.nn.BCEWithLogitsLoss()

trainer = Trainer(
    model=pipeline,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=trainloader,
    validdataloader=trainloader,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau,
    device=device,
)

In [ ]:
trainer.loop()

In [ ]:
import shutil

shutil.rmtree("tmp/", ignore_errors=True)
shutil.rmtree("logs/", ignore_errors=True)
shutil.rmtree("demo/", ignore_errors=True)